---
title: Training loop
numbering: false
---

# 0. 개요
+ 에폭 반복:
    + 학습 한 사이클 (train_loader, model.train(), 파라미터 업데이트)
    + 검증 한 사이클 (val_loader, model.eval() + torch.no_grad(), 성능 측정)
    + 검증 성능이 최고이면 체크포인트 저장
+ 마지막에 한 번 최고의 체크포인트로 테스트 세트 평가

# 1. 준비: 모델, 손실함수, optimizer

In [1]:
import torch
from torch import nn
from torch.utils.data import DataLoader, random_split
from torchvision import datasets
from torchvision.transforms import ToTensor

torch.manual_seed(0)
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")

full_train = datasets.FashionMNIST(root="data", train=True, download=True, transform=ToTensor())
test_ds = datasets.FashionMNIST(root="data", train=False, download=True, transform=ToTensor())
train_set, val_set = random_split(full_train, [50000, 10000], generator=torch.Generator().manual_seed(0))

train_loader = DataLoader(train_set, batch_size=64, shuffle=True, generator=torch.Generator().manual_seed(42))
val_loader = DataLoader(val_set, batch_size=256, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=256, shuffle=True)

class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Flatten(), # (B, 1, 28, 28) -> (B, 784)
            nn.Linear(784, 128),
            nn.ReLU(),
            nn.Linear(128, 10), # 클래스 10개에 대한 점수 logit
        )

    def forward(self, x):
        return self.net(x)

model = MLP().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

+ `nn.CrossEntropyLoss`
    + 입력은 softmax를 거치치 않은 그냥 점수(logit)을 넣어야 함. (B,10) 과 정수라벨 (B,), dtype int64
    + 내부에서 알아서 softmax 연산을 하기 때문에 모델 끝에 softmax를 호출해서 연산하면 학습이 망가진다
    + 출력은 배치 평균 손실, 스칼라 값이 나오게 된다.
+ `nn.Flatten()`
    + flatten(1)과 같은 동작을 한다. 배치 축을 남겨두고 나머지를 다 펼치게 된다.

# 2. 학습 한 사이클

In [2]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train() # train mode 켜기
    total_loss, n = 0.0, 0 # total loss 초기화

    for x, y in loader:
        x, y = x.to(device), y.to(device) # 배치 데이터도 모델과 같은 device 에 올린다
        logits = model(x) # forward pass
        loss = criterion(logits, y) # loss 계산

        optimizer.zero_grad() # gradient 초기화 (누적 방지)
        loss.backward() # gradient 계산
        optimizer.step() # gradient update

        total_loss += loss.item() * x.size(0) # loss 누적
        n += x.size(0)

    return total_loss / n


+ `loss.item()` : total_loss += loss 처럼 텐서를 그대로 더하면 배치의 computational graph가 연결된 채로 계속 쌓이게 된다. 메모리 누수 문제, 따라서 기록을 위한 숫자는 항상 .item()을 사용해서 꺼낸다
+ * x.size(0) 로 배치의 크기만큼 가중하기
    + loss 는 배치의 평균이다. 샘플 수를 곱해서 더한 다음 전체 샘플 수로 나눠주면 정확한 평균이 된다.

# 3. 평가 한 사이클

In [3]:
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss, correct, n = 0.0, 0, 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            logits = model(x)
            total_loss += criterion(logits, y).item() * x.size(0)
            correct += (logits.argmax(dim=1) == y).sum().item()
            n += x.size(0)
    return total_loss / n, correct / n

+ 평가 단계에서는 eval() 과 no_grad()를 둘 다 서야 한다
+ argmax(dim=1): logit (B, 10) 에서 클래스를 나타내는 1번축을 없애고 샘플마다 가장 높은 점수의 클래스의 번호를 고른다. 따라서 모양은 (B,)가 되고 이와 ==y 를 통해서 맞은 개수를 센다.

# 4. 에폭 반복 및 최고 성능 모델 저장

In [4]:
best_acc = 0.0

for epoch in range(3):
    train_loss = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, val_acc = evaluate(model, val_loader, criterion, device)
    print(f"epoch: {epoch} | train_loss : {train_loss:.4f} | val_loss : {val_loss:.4f} | val_acc : {val_acc:.4f}")

    if val_acc > best_acc: # 검증 성능이 최고일 때만 저장
        best_acc = val_acc
        torch.save(model.state_dict(), "best_model.pt")

model.load_state_dict(torch.load("best_model.pt", map_location=device))
test_loss, test_acc = evaluate(model, test_loader, criterion, device) # 테스트는 마지막에 단 한 번만 수행
print(f"test_acc : {test_acc:.4f}")

epoch: 0 | train_loss : 0.5648 | val_loss : 0.4317 | val_acc : 0.8440
epoch: 1 | train_loss : 0.4127 | val_loss : 0.4106 | val_acc : 0.8474
epoch: 2 | train_loss : 0.3702 | val_loss : 0.3748 | val_acc : 0.8613
test_acc : 0.8559


+ 최고의 모델을 고르는건 검증 세트 기반, 결과 보고는 테스트 세트 기반

# 5. 저장 및 불러오기

In [5]:
torch.save(model.state_dict(), "best_model.pt")
model.load_state_dict(torch.load("best_model.pt", map_location=device))

<All keys matched successfully>

In [6]:
ckpt = {
    "epoch": epoch,
    "model": model.state_dict(),
    "optimizer": optimizer.state_dict(),
    "best_acc": best_acc,
}

torch.save(ckpt, "checkpoint.pt")

# 이어서 학습하는 경우
ckpt = torch.load("checkpoint.pt", map_location=device)
model.load_state_dict(ckpt["model"])
optimizer.load_state_dict(ckpt["optimizer"])
start_epoch = ckpt["epoch"] + 1